# Model Building

**Table of contents**<a id='toc0_'></a>    
- [Config](#toc1_)    
  - [Setup](#toc1_1_)    
- [Data Processors](#toc2_)    
- [Loading Pre-trained LLM](#toc3_)    
- [Sentence Embeddings Treehouse](#toc4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Config](#toc0_)

In [ ]:
import os

DATA_DIR = "/kaggle/input/map-charting-student-math-misunderstandings"

X_COL_LABELS = ['row_id', 'QuestionText', 'MC_Answer', 'StudentExplanation']
Y_COL_LABELS = ['Category:Misconception', 'y_correct_response', 'y_misconception_exist', 'y_misconception_type']
COL_LABELS = X_COL_LABELS + Y_COL_LABELS

CORRECT_LABELS = [
    'True_Correct', # Correct Answer with Correct Understanding
    'True_Neither', # Correct Answer with Unidentifiable Misconception
    'True_Misconception' # Correct Answer with Identifiable Misconception
]

INCORRECT_LABELS = [
    'False_Misconception',
    'False_Neither',
    'False_Correct'
]


CATEGORY_LABELS = CORRECT_LABELS + INCORRECT_LABELS

CATEGORY_META_PROMPT = {
    "True_Correct": (
        "The student’s response is correct (i.e. they do not have a misconception). "
        "It’s 'True' meaning the ground-truth answer is valid. No conceptual error."
    ),
    "True_Neither": (
        "The student’s response is not incorrect in a way that maps to a known misconception, "
        "but also not exactly 'fully correct'. This may capture ambiguous or edge cases "
        "where the explanation doesn’t reveal correctness or a misconception clearly."
    ),
    "True_Misconception": (
        "The student’s answer is technically correct, but their explanation reveals a misconception. "
        "They may have arrived at the right answer by flawed reasoning or a partial misunderstanding."
    ),
    "False_Misconception": (
        "The student’s response is incorrect and the explanation indicates a specific misconception. "
        "This represents a clear case of misunderstanding tied to a known misconception category."
    ),
    "False_Neither": (
        "The student’s response is incorrect, but the explanation does not map to a known misconception. "
        "Their reasoning is wrong, but not in a way that matches the predefined misconception labels."
    ),
    "False_Correct": (
        "The student’s response is incorrect, but their explanation shows correct reasoning. "
        "This likely reflects a minor slip or arithmetic mistake rather than a conceptual error."
    )
}


## <a id='toc1_1_'></a>[Setup](#toc0_)

In [4]:
import pandas as pd

In [10]:

def load_train_dataset(path: str = DATA_DIR + "/train.csv") -> pd.DataFrame:

    train_data = pd.read_csv(path)
    train_data.fillna('NA', inplace=True)

    train_data['Category:Misconception'] = train_data['Category'] + ":" + train_data['Misconception']

    train_data['y_correct_response'] = train_data['Category'].str.split('_').str[0]
    train_data['y_misconception_exist'] = train_data['Category'].str.split('_').str[-1]
    train_data['y_misconception_type'] = train_data['Misconception']

    cat_file = train_data['Category:Misconception'].unique().tolist()
    return train_data[COL_LABELS], cat_file

train_data, CAT_ALL_LABELS = load_train_dataset()

In [11]:
train_data

,row_id,QuestionText,MC_Answer,StudentExplanation,Category:Misconception,y_correct_response,y_misconception_exist,y_misconception_type
0,0,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),0ne third is equal to tree nineth,True_Correct:NA,True,Correct,NA
1,1,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),1 / 3 because 6 over 9 is 2 thirds and 1 third...,True_Correct:NA,True,Correct,NA
2,2,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),"1 3rd is half of 3 6th, so it is simplee to un...",True_Neither:NA,True,Neither,NA
3,3,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),1 goes into everything and 3 goes into nine,True_Neither:NA,True,Neither,NA
4,4,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),1 out of every 3 isn't coloured,True_Correct:NA,True,Correct,NA
...,...,...,...,...,...,...,...,...
36691,36691,The probability of an event occurring is \( 0....,Unlikely,very unlikely.,False_Neither:NA,False,Neither,NA
36692,36692,The probability of an event occurring is \( 0....,Unlikely,yea pretty much unlikely from what it seems.,False_Neither:NA,False,Neither,NA
36693,36693,The probability of an event occurring is \( 0....,Unlikely,yeah pretty unlikelyy from what it seems.,False_Neither:NA,False,Neither,NA
36694,36694,The probability of an event occurring is \( 0....,Unlikely,you have 9 other numbers to get.,False_Neither:NA,False,Neither,NA


# <a id='toc2_'></a>[Data Processors](#toc0_)

In [22]:
from typing import Literal
from dataclasses import dataclass, field

@dataclass
class PredLabel:
    _correct: bool = False
    _misconception_exist: Literal["Neither", "Correct", "Misconception"] | None = None
    _misconception_type: str | None = None
    pred_labels: dict[str, float] = field(default_factory=lambda: {cat: 0.0 for cat in CAT_ALL_LABELS})

    @property
    def correct(self):
        return "True" if self._correct is True else "False"

    @correct.setter
    def correct(self, correct_answer: bool):
        self._correct = correct_answer
        label = "True" if correct_answer is True else "False"

        self.pred_labels = {key: val + 1 if label in key else val for key, val in self.pred_labels.items()}

    @property
    def misconception_exist(self):
        return self._misconception_exist

    @misconception_exist.setter
    def misconception_exist(self, misconception: Literal["Neither", "Correct", "Misconception"]):
        # Neither means not in misconception labels trained
        # Correct means explanation shows no misconception
        self._misconception_exist = misconception
        # label = self.correct + "_" + misconception
        correct_label = self.correct
        # all_cats = {"Neither", "Correct", "Misconception"}
        # miscon_remaining = all_cats - {misconception}

        pred_labels = self.pred_labels.copy()
        other_labels = [i for i in ['Neither', 'Correct', 'Misconception'] if i != misconception]

        for key, val in pred_labels.items():
            if correct_label in key and misconception not in key:
                if key in other_labels:
                    v = val - 0.5
                else:
                    v = val - 1
                self.pred_labels[key] = v
            elif correct_label in key and misconception in key:
                self.pred_labels[key] += 0.5

    @property
    def misconception_type(self):
        return self._misconception_type

    @misconception_type.setter
    def misconception_type(self, label: str):
        self._misconception_type = label
        label = self.correct + "_" + self.misconception_exist + ":" + label
        val = self.pred_labels.get(label, 0.0)

        self.pred_labels.update({
            label: val + 1
        })

    def to_submission_label(self):
        if isinstance(self.misconception_exist, str) and self.misconception_exist == "Misconception":
            raise ValueError("Cannot run final prediction labels. Misconception must be configured if misconception identified.")

        # Step 1: sort all labels by score
        sorted_items = sorted(self.pred_labels, key=lambda x: x[1], reverse=True)
        print("sorted_items", sorted_items)
        return " ".join(sorted_items[:3])

# DEMO TEST
pred = PredLabel()
# pred MC_Answer is correct
pred.correct = True
# pred Misconception exists
pred.misconception_exist = "Correct"
# pred Misconception type if exists
# pred.misconception_type = "NA"
pred.to_submission_label()

sorted_items ['True_Correct:NA', 'True_Neither:NA', 'True_Misconception:Incomplete', 'True_Misconception:WNB', 'True_Misconception:SwapDividend', 'True_Misconception:Mult', 'True_Misconception:FlipChange', 'True_Misconception:Irrelevant', 'True_Misconception:Additive', 'True_Misconception:Not_variable', 'True_Misconception:Inversion', 'True_Misconception:Duplication', 'True_Misconception:Whole_numbers_larger', 'True_Misconception:Shorter_is_bigger', 'True_Misconception:Longer_is_bigger', 'True_Misconception:Wrong_fraction', 'True_Misconception:Adding_across', 'True_Misconception:Denominator-only_change', 'True_Misconception:Incorrect_equivalent_fraction_addition', 'True_Misconception:Division', 'True_Misconception:Subtraction', 'True_Misconception:Definition', 'True_Misconception:Tacking', 'True_Misconception:Positive', 'True_Misconception:Wrong_term', 'True_Misconception:Firstterm', 'True_Misconception:Base_rate', 'True_Misconception:Multiplying_by_4', 'False_Neither:NA', 'False_Misco

'True_Correct:NA True_Neither:NA True_Misconception:Incomplete'

# <a id='toc3_'></a>[Loading Pre-trained LLM](#toc0_)

In [50]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M-Instruct")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

I'm [Your Name]! I'm a software engineer at [Your Company/Organization]. I'm excited to share my passion for [specific topic or industry]. I'm always looking for ways to


In [53]:
# PREPARATION PROMPT
SYSTEM_PROMPT = """
You are a helpful assistant. When writing mathematical expressions, always format them as double-escaped LaTeX strings suitable for embedding inside Python or JSON.

- Wrap inline math in '\\(' and '\\)'.
- Escape every backslash with another backslash.
- Example: Instead of writing "1/3", output '\\( \\frac{1}{3} \\)'.
- Always return expressions in this form, e.g. '\\( x^2 + y^2 = z^2 \\)'.

Do not render math in plain text or Markdown, only in escaped LaTeX string form.
"""

ANTI_SYSTEM_PROMPT = """
You are an assistant that always responds with INCORRECT mathematical answers.
All answers must be formatted as double-escaped LaTeX strings suitable for Python or JSON.

Formatting rules:
- Wrap all math in '\\(' and '\\)'.
- Escape every backslash with another backslash.
- Example: Instead of giving the correct fraction 1/3, you might output '\\( \\frac{2}{3} \\)' or '\\( 5 \\)'.
- Do not explain or justify why it is wrong — simply provide the incorrect expression.
- Do not output plain text or Markdown. Always return only one escaped LaTeX string as the answer.
"""

# MISCONCEPTION EXISTENCE PROMPTS
STUDENT_RESPONSE_NARRATOR = """
You are an assistant summarizing the student's explanation for their selected response in a maths exam. Your task is to clean up the response and reformat in a third-person point of view that highlights the student's misconception within their response.
"""

MODEL_CONFIG = dict(
    # setup config
    expert_system_prompt=SYSTEM_PROMPT,
    anti_system_prompt=ANTI_SYSTEM_PROMPT,
    # device="mps",
    # tokenizer config
    add_generation_prompt=True,
    # generation config
    max_new_tokens=512,

)

class Basement:
    def __init__(
        self,
        model,
        tokenizer,
        **kwargs
    ):
        self.model = model
        self.tokenizer = tokenizer

        if kwargs:
            print(f"Updating model configuration: {kwargs}")
            MODEL_CONFIG.update(**kwargs)

        for key, val in MODEL_CONFIG.items():
            setattr(self, f"_{key}", val)

    def format_message(self, system_prompt: str, user_input: str, return_ids: bool = True) -> list[dict] | str:
        msg = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]

        text = self.tokenizer.apply_chat_template(
            msg,
            tokenize=False,
            add_generation_prompt=self._add_generation_prompt # pyright: ignore[reportAttributeAccessIssue]
        )

        return text if return_ids is False else self.tokenizer([text], return_tensors="pt")

    def _generate_text(
        self,
        system_prompt: str,
        user_input: str,
    ):
        print(f'Passing system prompt: {system_prompt}')

        model_inputs = self.format_message(
            system_prompt=system_prompt,
            user_input=user_input
        )

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self._max_new_tokens
        )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        return self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    def correct_response(self, question: str):
        return self._generate_text(
            system_prompt=self._expert_system_prompt,
            user_input=question
        )

    def incorrect_response(self, question: str):
        return self._generate_text(
            system_prompt=self._anti_system_prompt,
            user_input=question
        )


basement = Basement(model=model, tokenizer=tokenizer)

In [ ]:
train_data.map()

,row_id,QuestionText,MC_Answer,StudentExplanation,Category:Misconception,y_correct_response,y_misconception_exist,y_misconception_type
0,0,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),0ne third is equal to tree nineth,True_Correct:NA,True,Correct,NA
1,1,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),1 / 3 because 6 over 9 is 2 thirds and 1 third...,True_Correct:NA,True,Correct,NA
2,2,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),"1 3rd is half of 3 6th, so it is simplee to un...",True_Neither:NA,True,Neither,NA
3,3,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),1 goes into everything and 3 goes into nine,True_Neither:NA,True,Neither,NA
4,4,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),1 out of every 3 isn't coloured,True_Correct:NA,True,Correct,NA
...,...,...,...,...,...,...,...,...
36691,36691,The probability of an event occurring is \( 0....,Unlikely,very unlikely.,False_Neither:NA,False,Neither,NA
36692,36692,The probability of an event occurring is \( 0....,Unlikely,yea pretty much unlikely from what it seems.,False_Neither:NA,False,Neither,NA
36693,36693,The probability of an event occurring is \( 0....,Unlikely,yeah pretty unlikelyy from what it seems.,False_Neither:NA,False,Neither,NA
36694,36694,The probability of an event occurring is \( 0....,Unlikely,you have 9 other numbers to get.,False_Neither:NA,False,Neither,NA


Improvements:

- Additional Preprocessors to condition the feature vectors
  
```python
def child_role_play():
    pass

def add_contrast_responses():
    pass

def build_feature_pool(row):
    pass

```

# <a id='toc4_'></a>[Sentence Embeddings Treehouse](#toc0_)

In [66]:
from sentence_transformers import SentenceTransformer, util

SENTENCE_MODEL_ID = "all-MiniLM-L6-v2"
pooler = SentenceTransformer(SENTENCE_MODEL_ID)

In [91]:
import torch
import numpy as np
from collections import namedtuple
from dataclasses import dataclass, field

Obs = namedtuple('Obs', ['correct', 'incorrect'])

HIDDEN_DIM = 384

FACT_INPUT_LABELS = [
    "QuestionId",
    "QuestionText"
    "CorrectResponse",
    "CorrectExplanation",
    "IncorrectResponse",
]

FACT_OUTPUT_LABELS = ['CorrectResponseVec', 'ContrastPair']  #'CorrectExplanationVec',

# observed input features
OBS_INPUT_LABELS = [
    "row_id"
    "QuestionId",
    "MC_Answer"
]

# labels that are returned from fn methods during runtime
OBS_OUTPUT_LABELS = ['PredFact', 'PredObs']

@dataclass
class ObserveHistory:
    correct: list = field(default_factory=list)
    incorrect: list = field(default_factory=list)

    @property
    def to_numpy(self):
        return (np.vstack(self.correct), np.vstack(self.incorrect))

    @property
    def to_tensor(self):
        """Convert lists or numpy arrays into stacked PyTorch tensors."""
        def to_tensor_safe(data):
            if not data:
                return torch.empty((0,))
            # Handle mixed input types (list of lists, list of arrays, single array)
            if isinstance(data, np.ndarray):
                arr = data
            else:
                # Convert to array first to handle nested lists safely
                arr = np.array(data, dtype=np.float32)
                if arr.ndim == 1:
                    arr = np.expand_dims(arr, 0)
            return torch.as_tensor(arr, dtype=torch.float32)

        correct_tensor = to_tensor_safe(self.correct)
        incorrect_tensor = to_tensor_safe(self.incorrect)

        return correct_tensor, incorrect_tensor

@dataclass
class FactTable:
    # data input
    qid: int
    question_text: str
    # prompt output
    correct_response: str
    correct_explanation: str
    incorrect_response: str
    incorrect_explanation: str
    # sentence encoder output
    correct_response_vec: np.ndarray
    contrast_pair: np.ndarray
    observed: ObserveHistory = field(default_factory=ObserveHistory)

@dataclass
class Observation:
    # data input
    row_id: int
    qid: int
    question_text: str
    mc_answer: str
    student_explanation: str

class Gatekeeper:
    # tracks fact table, observations, positive/negative embeddings

    def __init__(
        self,
        model: SentenceTransformer,
        threshold: float = 0.5,
        obs_threshold: float = 0.5
    ):
        self.model = model

        # agent's fact table
        self.fact_table = {}
        # to avoid different rep of correct answer
        # self.obs_insights = {}

        self.threshold = threshold
        self.obs_threshold = obs_threshold

        self.input_fact_labels = FACT_INPUT_LABELS
        self.output_fact_labels = FACT_OUTPUT_LABELS
        self.fact_labels = self.input_fact_labels + self.output_fact_labels
        self.input_obs_labels = OBS_INPUT_LABELS
        self.output_obs_labels = OBS_OUTPUT_LABELS
        self.obs_labels = self.input_obs_labels + self.output_obs_labels

    def _pred_with_fact_table(self, qid: str, mc_answer: np.ndarray) -> None | np.ndarray | torch.Tensor:

        if fact := self.fact_table.get(qid, None):
            pos_neg = fact.contrast_pair

            if isinstance(pos_neg, np.ndarray):

                pred = util.pytorch_cos_sim(mc_answer, pos_neg) # type: ignore
                assert pred.shape[-1] == pos_neg.shape[0] and len(pred.shape) == 2, f'Fact table expected (1, {HIDDEN_DIM}) shape for contrasting pairs. Instead, output shape: {pred.shape} with length pred shape: {len(pred.shape)}. Input has shape: {mc_answer.shape} and is dtype {type(mc_answer)}. Fact table shape is {pos_neg.shape}'

                proba = pred / pred.sum()
                # return dict(zip(('correct', 'incorrect'), proba))
                return proba
            else:
                raise ValueError(f'Unexpected type saved for embedding: {pos_neg}')

        return None

    def _pred_with_observed(self, qid: str, mc_answer: np.ndarray):
        if fact_table := self.fact_table.get(qid, None):

            insights = fact_table.observed
            insights_correct, insights_incorrect = insights.to_numpy
            insights_correct = torch.from_numpy(insights_correct)
            insights_incorrect = torch.from_numpy(insights_incorrect)
            mean_pos_prob = mean_neg_prob = None

            if insights_correct.shape[0] > 0:
                pos_pred = util.pytorch_cos_sim(mc_answer, insights_correct)
                assert len(pos_pred.shape) == 2 and pos_pred.shape[-1] == insights_correct.shape[0]
                pos_pred = pos_pred.squeeze(0) if not isinstance(pos_pred, np.ndarray) else pos_pred
                mean_pos_prob = pos_pred.min()

                print('Pos prob mu: ', pos_pred.mean())
                print('Pos prob max: ', pos_pred.max())
                print('Pos prob min: ', pos_pred.min())

            if insights_incorrect.shape[0] > 0:
                neg_pred = util.pytorch_cos_sim(mc_answer, insights_incorrect)
                assert len(neg_pred.shape) == 2 and neg_pred.shape[-1] == insights_incorrect.shape[0]
                mean_neg_prob = neg_pred.min()
                neg_pred = neg_pred.squeeze(0) if not isinstance(neg_pred, np.ndarray) else neg_pred
                print('Neg prob mu: ', neg_pred.mean())
                print('Neg prob max: ', neg_pred.max())
                print('Neg prob min: ', neg_pred.min())

            if mean_pos_prob and mean_neg_prob:
                proba = np.array([mean_pos_prob.numpy(), mean_neg_prob.numpy()])
                prob = proba / proba.sum()

                if prob.argmax() == 0:
                    self.fact_table[qid].observed.correct.append(mc_answer)
                else:
                    self.fact_table[qid].observed.incorrect.append(mc_answer)

                #return dict(zip(('correct', 'incorrect'), prob))
                return prob

        print(f'Returning no pred from observations. Either: \n- Cannot find QuestionId {qid} in fact_table or obs_insights. Add to fact_table or obs_insights to initiate. OR\n- Error while normalizing probas.')
        return None

    def predict(self, qid: str, mc_answer: str | np.ndarray):
        assert qid in self.fact_table, f'Cannot find QuestionId {qid} in fact_table or obs_insights. Add to fact_table or obs_insights to initiate.'

        mc_answer_vec = self.model.encode([mc_answer]) if isinstance(mc_answer, str) else mc_answer

        z0 = self._pred_with_fact_table(qid, mc_answer_vec)
        z1 = self._pred_with_observed(qid, mc_answer_vec)
        print(f"Output Fact Table Pred: {z0}\nOutput Obs Pred: {z1}\n")

        if not (isinstance(z0, (torch.Tensor, np.ndarray)) and isinstance(z1, (torch.Tensor, np.ndarray))):
            raise TypeError(f'z0 and z1 must be tensors or numpy arrays. Got z0={type(z0)}, z1={type(z1)}')

        z0_margin = np.diff(z0) if isinstance(z0, np.ndarray) else torch.diff(z0)
        z1_margin = np.diff(z1) if isinstance(z1, np.ndarray) else torch.diff(z1)
        margin_ratio = z0_margin / z1_margin

        print(f"Output Fact Table Pred: {z0}\tMargin {z0_margin}\nOutput Obs Pred: {z1}\tMargin {z1_margin}\nRatio (Fact/Obs): {margin_ratio}\nCertainity {1 - margin_ratio}")

        return {
            "pred_fact_prob": z0,
            "pred_obs_prob": z1,
            "fact_margin": z0_margin,
            "obs_margin": z1_margin,
            "margin_ratio": margin_ratio
        }

    def add_fact(self, **kwargs):
        assert set(kwargs.keys()) != set(self.input_fact_labels), f'Expected all of features in fact table to be defined, only have: {set(kwargs.keys()) - set(self.input_fact_labels)}'

        qid = kwargs.get('QuestionId', None)
        correct_answer = kwargs.get('CorrectResponse', None)
        correct_explanation = kwargs.get('CorrectExplanation', None)
        incorrect_answer = kwargs.get('IncorrectResponse', None)
        incorrect_explanation = kwargs.get('IncorrectExplanation', None)

        assert isinstance(correct_answer, str) and isinstance(correct_explanation, str)

        correct_answer_v = self.model.encode([f"Answer: {correct_answer}\n" + f"Explanation: {correct_explanation}"])
        incorrect_answer_v = self.model.encode([f"Answer: {incorrect_answer}\n" + f"Explanation: {incorrect_explanation}"])
        assert tuple(correct_answer_v.shape) == tuple((1, HIDDEN_DIM)) and tuple(incorrect_answer_v.shape) == tuple((1, HIDDEN_DIM))

        pro_neg = np.vstack([correct_answer_v, incorrect_answer_v])
        assert tuple(pro_neg.shape) == tuple((2, HIDDEN_DIM))
        # initialise
        self.fact_table[qid] = FactTable(
            qid=qid,
            question_text=kwargs.get('QuestionText'),
            correct_response=correct_answer,
            correct_explanation=correct_explanation,
            incorrect_response=incorrect_answer,
            incorrect_explanation=incorrect_explanation,
            correct_response_vec=correct_answer_v,
            contrast_pair=pro_neg,
            observed=ObserveHistory(correct=[correct_answer_v], incorrect=[incorrect_answer_v])

        )

gate = Gatekeeper(model=pooler)

sample_fact = {
    "QuestionId": "Q1",
    "QuestionText": 'What fraction of the shape is not shaded? Give your answer in its simplest form. [Image: A triangle split into 9 equal smaller triangles. 6 of them are shaded.]',
    "CorrectResponse": "\\( \\frac{1}{3} \\)",
    "CorrectExplanation": "There are 9 equal parts in total. 6 are shaded and 3 are not shaded. The fraction not shaded is \\( \\tfrac{3}{9} \\), which simplifies to \\( \\tfrac{1}{3} \\).",
    "IncorrectResponse":  "\\( \\frac{6}{9} \\)",
    "IncorrectExplanation": "There are 9 equal parts in total. 6 are shaded and 3 are not shaded."
    # "IncorrectExplanation": "There are 6 shaded parts out of 9 in total. So the fraction not shaded is \\( \\tfrac{6}{9} \\), which simplifies to \\( \\tfrac{2}{3} \\). "
}

gate.add_fact(**sample_fact)

# True_Correct, False_Misconception/Neither, False_Correct
sample_output = [gate.predict(qid="Q1", mc_answer="\\( \\frac{1}{3} \\) There are 9 equal parts in total. 6 are shaded and 3 are not shaded."), gate.predict(qid="Q1", mc_answer="\\( \\frac{2}{3} \\). There are 6 shaded parts out of 9 in total. So the fraction not shaded is \\( \\tfrac{6}{9} \\), which simplifies to \\( \\tfrac{2}{3} \\)."), gate.predict(qid="Q1", mc_answer="4/5. There are 9 equal parts in total. 6 are shaded and 3 are not shaded. The fraction not shaded is \\( \\tfrac{3}{9} \\)")]
print('Obs shape:', len(gate.fact_table['Q1'].observed.correct), 'and incorrect:', len(gate.fact_table['Q1'].observed.incorrect))

Pos prob mu:  tensor(0.9499)
Pos prob max:  tensor(0.9499)
Pos prob min:  tensor(0.9499)
Neg prob mu:  tensor(0.9381)
Neg prob max:  tensor(0.9381)
Neg prob min:  tensor(0.9381)
Output Fact Table Pred: tensor([[0.5031, 0.4969]])
Output Obs Pred: [0.5031112  0.49688882]

Output Fact Table Pred: tensor([[0.5031, 0.4969]])	Margin tensor([[-0.0062]])
Output Obs Pred: [0.5031112  0.49688882]	Margin [-0.00622237]
Ratio (Fact/Obs): tensor([[1.]])
Certainity tensor([[0.]])
Pos prob mu:  tensor(0.9147)
Pos prob max:  tensor(0.9460)
Pos prob min:  tensor(0.8835)
Neg prob mu:  tensor(0.8794)
Neg prob max:  tensor(0.8794)
Neg prob min:  tensor(0.8794)
Output Fact Table Pred: tensor([[0.5182, 0.4818]])
Output Obs Pred: [0.50116235 0.49883768]

Output Fact Table Pred: tensor([[0.5182, 0.4818]])	Margin tensor([[-0.0365]])
Output Obs Pred: [0.50116235 0.49883768]	Margin [-0.00232467]
Ratio (Fact/Obs): tensor([[15.6839]])
Certainity tensor([[-14.6839]])
Pos prob mu:  tensor(0.9144)
Pos prob max:  tenso

In [81]:
np.mean(np.diff(([[2, 2]])))

0.0

## 